# Choice set generation

Graph based

In [ ]:
import geopandas as gpd
import networkx as nx
import zipfile
import pandas as pd

# --- Unzip the provided file ---
zip_file_path = "Sydney_CBD_SA1.zip"
unzip_dir = "Sydney_CBD_SA1_unzipped"

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(unzip_dir)

# --- Load the shapefile ---
gdf = gpd.read_file(f"{unzip_dir}/Sydney_CBD_SA1.shp")
gdf = gdf.to_crs("EPSG:3577")  # Use projected CRS for spatial operations

# --- Step 1: Create an empty NetworkX graph ---
graph = nx.Graph()

# --- Step 2: Add nodes ---
for idx, row in gdf.iterrows():
    graph.add_node(row["SA1_CODE21"], geometry=row.geometry)

# --- Step 3: Efficiently add edges using spatial index (R-tree) ---
sindex = gdf.sindex

for idx, row in gdf.iterrows():
    sa1_code = row["SA1_CODE21"]
    geom = row.geometry
    
    # Get candidate neighbors using bounding box
    candidate_idxs = list(sindex.intersection(geom.bounds))
    for cand_idx in candidate_idxs:
        if idx >= cand_idx:
            continue  # avoid duplicate and self-pairs

        cand_row = gdf.iloc[cand_idx]
        cand_code = cand_row["SA1_CODE21"]
        
        if geom.touches(cand_row.geometry):
            graph.add_edge(sa1_code, cand_code)

# --- Step 4: Extract reachable nodes within 2 hops (including hop = 0) ---
data = []

for node in graph.nodes:
    hop_lengths = nx.single_source_shortest_path_length(graph, node, cutoff=2)
    for reachable_node, hop in hop_lengths.items():
        data.append({
            "Node": node,
            "Reachable_Node": reachable_node,
            "Hop": hop
        })

# --- Step 5: Create DataFrame ---
df_reachable = pd.DataFrame(data)

# --- Output ---
print(df_reachable.head())
print(f"\nTotal reachable pairs (including hop=0): {len(df_reachable)}")

Network distance based

In [ ]:
import geopandas as gpd
import pandas as pd
import networkx as nx
import zipfile

# --- Unzip the provided file ---
zip_file_path = "Sydney_CBD_SA1.zip"
unzip_dir = "Sydney_CBD_SA1_unzipped"

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(unzip_dir)

# --- Load shapefile and reproject for metric distances ---
sa1_gdf = gpd.read_file(f"{unzip_dir}/Sydney_CBD_SA1.shp")
sa1_gdf = sa1_gdf.to_crs("EPSG:3577")  # Australian Albers projection
sa1_gdf['centroid'] = sa1_gdf.geometry.centroid

# --- Build graph ---
graph = nx.Graph()

# Add nodes with centroid positions
for idx, row in sa1_gdf.iterrows():
    graph.add_node(row['SA1_CODE21'], pos=(row['centroid'].x, row['centroid'].y))

# --- Efficient edge creation using spatial index ---
threshold = 2000  # meters
sindex = sa1_gdf.sindex

for idx, row in sa1_gdf.iterrows():
    sa1_code = row['SA1_CODE21']
    centroid = row['centroid']
    buffer = centroid.buffer(threshold)

    # Get nearby centroid candidates using spatial index
    candidate_idxs = list(sindex.intersection(buffer.bounds))
    
    for cand_idx in candidate_idxs:
        if idx >= cand_idx:
            continue  # Avoid self and duplicate comparisons

        cand_row = sa1_gdf.iloc[cand_idx]
        cand_code = cand_row['SA1_CODE21']
        cand_centroid = cand_row['centroid']

        dist = centroid.distance(cand_centroid)
        if dist <= threshold:
            graph.add_edge(sa1_code, cand_code, weight=dist)


# List to store the results
reachable_sa1s_list = []
# Define unique origins from SA1 codes
unique_origins = sa1_gdf['SA1_CODE21'].unique()

# Iterate through each unique origin SA1 code
for origin in unique_origins:
    if origin in graph.nodes:
        # Compute shortest path lengths from the origin within 2000 meters
        lengths = nx.single_source_dijkstra_path_length(graph, origin, cutoff=2000)
        
        # Append results to the list (include the origin itself with distance 0)
        for dest, distance in lengths.items():
            reachable_sa1s_list.append({
                'Origin SA1': origin,
                'Network Distance (meters)': distance
            })

# Convert the results to a DataFrame
reachable_sa1s_df = pd.DataFrame(reachable_sa1s_list)



# Display the resulting DataFrame
print(reachable_sa1s_df)


Hybrid approach

In [ ]:
import geopandas as gpd
import networkx as nx
import zipfile
import pandas as pd

# --- Parameters ---
D_MAX = 2000  # max network distance in meters
HOP_MAX = 2   # max number of hops

# --- Unzip and Load Shapefile ---
zip_file_path = "Sydney_CBD_SA1.zip"
unzip_dir = "Sydney_CBD_SA1_unzipped"

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(unzip_dir)

gdf = gpd.read_file(f"{unzip_dir}/Sydney_CBD_SA1.shp")
gdf = gdf.to_crs("EPSG:3577")  # Use projected CRS for accurate distances in meters
gdf["centroid"] = gdf.geometry.centroid

# --- Build Graph Using Spatial Index ---
graph = nx.Graph()

# Add all nodes with centroids
for idx, row in gdf.iterrows():
    graph.add_node(row["SA1_CODE21"], centroid=row["centroid"])

# Build spatial index
sindex = gdf.sindex

# Efficient edge creation using spatial index + touches
for idx, row in gdf.iterrows():
    sa1_code = row["SA1_CODE21"]
    geom = row.geometry
    centroid = row["centroid"]

    # Get potential touching neighbors using bounding box intersection
    candidate_idxs = list(sindex.intersection(geom.bounds))
    for cand_idx in candidate_idxs:
        if idx >= cand_idx:
            continue  # Avoid duplicate pairs

        candidate = gdf.iloc[cand_idx]
        candidate_code = candidate["SA1_CODE21"]

        if geom.touches(candidate.geometry):
            dist = centroid.distance(candidate["centroid"])
            graph.add_edge(sa1_code, candidate_code, weight=dist)

# --- Extract Hybrid Reachable Set (Includes hop = 0) ---
data = []

for origin in graph.nodes:
    # 1. Network distance ≤ 2km
    dist_lengths = nx.single_source_dijkstra_path_length(graph, origin, cutoff=D_MAX, weight="weight")

    # 2. Hop-based reachability ≤ 2 hops
    hop_lengths = nx.single_source_shortest_path_length(graph, origin, cutoff=HOP_MAX)

    # 3. Intersection = hybrid condition (includes hop = 0)
    hybrid_nodes = set(dist_lengths.keys()) & set(hop_lengths.keys())

    for dest in hybrid_nodes:
        data.append({
            "Origin SA1": origin,
            "Available Destination SA1": dest,
            "Hop": hop_lengths[dest],
            "Network_Distance": dist_lengths[dest]
        })

# --- Create DataFrame ---
df_reachable = pd.DataFrame(data)

# --- Output ---
print(df_reachable.head())
print(f"\nTotal hybrid reachable pairs (including hop=0): {len(df_reachable)}")

df_reachable['Network_Distance_replaced'] = df_reachable['Network_Distance'].replace(0, 400)


# Sydney OD Transferability 

In [1]:
import pandas as pd

sydney_df=pd.read_csv('destination_choice_processed_SA1_SYD_hybrid_hop2_network_2KM_NegBin_All_Seattle.csv')

In [4]:
import networkx as nx

def add_graph_topology_features(df):
    df = df.copy()

    G = nx.DiGraph()
    for _, row in df.iterrows():
        G.add_edge(row['ORIGSA1_2021'], row['DESTSA1_2021'])
    G.remove_edges_from(nx.selfloop_edges(G))

    degree     = dict(G.degree())
    closeness  = nx.closeness_centrality(G)
    betweenness = nx.betweenness_centrality(G)
    pagerank   = nx.pagerank(G)
    core       = nx.core_number(G)

    # Origin-side
    df['origin_degree']     = df['ORIGSA1_2021'].map(degree)
    df['origin_closeness']  = df['ORIGSA1_2021'].map(closeness)
    df['origin_betweenness']= df['ORIGSA1_2021'].map(betweenness)
    df['origin_pagerank']   = df['ORIGSA1_2021'].map(pagerank)
    df['origin_core']       = df['ORIGSA1_2021'].map(core)

    # Destination-side
    df['dest_degree']       = df['DESTSA1_2021'].map(degree)
    df['dest_closeness']    = df['DESTSA1_2021'].map(closeness)
    df['dest_betweenness']  = df['DESTSA1_2021'].map(betweenness)
    df['dest_pagerank']     = df['DESTSA1_2021'].map(pagerank)
    df['dest_core']         = df['DESTSA1_2021'].map(core)

    return df


In [5]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def predict_TLGnet(MODEL_NAME, df_new, HD, device):

    # ---------------- MODEL ----------------
    class FlowPredictor(torch.nn.Module):
        def __init__(self, input_dim, hidden_dim):
            super().__init__()
            self.fc1 = torch.nn.Linear(input_dim, hidden_dim)
            self.fc2 = torch.nn.Linear(hidden_dim, 1)
        def forward(self, x):
            return self.fc2(F.relu(self.fc1(x))).squeeze(-1)

    # ------------- Sparsemax ---------------
    class Sparsemax(torch.nn.Module):
        def __init__(self, dim=None):
            super().__init__()
            self.dim = dim
        def forward(self, input):
            input = input - input.max(dim=self.dim, keepdim=True)[0]
            zs = torch.sort(input, descending=True, dim=self.dim)[0]
            r = torch.arange(1, zs.size(self.dim)+1, dtype=input.dtype, device=input.device)
            v = [1]*input.dim(); v[self.dim] = -1
            r = r.view(v)
            cumsum = zs.cumsum(dim=self.dim)
            bound = 1 + r * zs
            is_gt = bound > cumsum
            k = is_gt.sum(dim=self.dim, keepdim=True)
            tau = (cumsum.gather(self.dim, k-1) - 1) / k
            return torch.clamp(input - tau, min=0)

    # Load model
    model = FlowPredictor(input_dim=34, hidden_dim=HD).to(device)
    model.load_state_dict(torch.load(MODEL_NAME, map_location=device))
    model.eval()

    # Feature list used in training
    features = [
        'Hop','Population_origin','Population_destination','stations',
        'Number of Links/Number of Nodes','Intersection Count','Commercial',
        'Education','Industrial','Parkland','Transport','Water',
        'stations_origin','Number of Links/Number of Nodes_origin',
        'Intersection Count_origin','Commercial_origin','Education_origin',
        'Industrial_origin','Parkland_origin','Transport_origin','Water_origin',
        'Household median age','Weekly household income','DISTANCE',
        'origin_degree','dest_degree','origin_closeness','dest_closeness',
        'origin_betweenness','dest_betweenness','origin_pagerank','dest_pagerank',
        'origin_core','dest_core'
    ]

    # Prepare scaled features
    X = df_new[features].replace([np.inf,-np.inf],np.nan).fillna(0).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)

    # Build indices
    origin_to_idx = {o:i for i,o in enumerate(df_new['ORIGSA1_2021'].unique())}
    dest_to_idx   = {d:i for i,d in enumerate(df_new['DESTSA1_2021'].unique())}

    df_new['origin_idx'] = df_new['ORIGSA1_2021'].map(origin_to_idx)
    df_new['destination_idx'] = df_new['DESTSA1_2021'].map(dest_to_idx)

    # Raw model scores
    with torch.no_grad():
        scores = model(X_tensor)

    # Sparsemax normalization per origin
    sparsemax = Sparsemax(dim=0)
    scores_by_origin = torch.zeros((len(origin_to_idx), len(dest_to_idx)), device=device)

    for o, idx_o in origin_to_idx.items():
        mask = (df_new['origin_idx'] == idx_o).values
        idxs = np.where(mask)[0]

        scores_o = scores[idxs]
        dest_ids = torch.tensor(df_new.iloc[idxs]['destination_idx'].values, device=device)

        probs_o = sparsemax(scores_o)
        scores_by_origin[idx_o].scatter_(0, dest_ids, probs_o)

    # Total flows per origin
    df_new['O_i'] = df_new.groupby('origin_idx')['Weighted_Trips_SA1'].transform('sum')
    O_i = torch.tensor(df_new['O_i'].values, dtype=torch.float32).reshape(-1,1).to(device)

    # Final values
    origin_idx = df_new['origin_idx'].values
    dest_idx   = df_new['destination_idx'].values

    pred_probs = scores_by_origin[origin_idx, dest_idx].reshape(-1,1)
    pred_flows = pred_probs * O_i

    # Output dataframe
    result = pd.DataFrame({
        "ORIGSA1_2021": df_new["ORIGSA1_2021"],
        "DESTSA1_2021": df_new["DESTSA1_2021"],
        "Predicted_Prob": pred_probs.cpu().numpy().flatten(),
        "Predicted_Flow": pred_flows.cpu().numpy().flatten()
    })

    return result


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Step 1: Add graph topology features
df_sydney_with_features = add_graph_topology_features(sydney_df)


# Step 2: Predict flows
predicted_df = predict_TLGnet(
    MODEL_NAME="Seattle_revised_0.01_500.pt",
    # MODEL_NAME="TLGNet_MEL_0.01_500.pt",
    df_new=df_sydney_with_features,
    HD=128,
    device=device
)

predicted_df.head()


/var/folders/2w/2667g69j0632m4l5mjnrj3hr0000gn/T/ipykernel_2195/890163012.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_NAME, m

,ORIGSA1_2021,DESTSA1_2021,Predicted_Prob,Predicted_Flow
0,11701163404,11701163404,0.000000,0.000000
1,11701163404,11701163410,0.093796,39.832985
2,11701163404,11701163420,0.000000,0.000000
3,11701163404,11703164303,0.000000,0.000000
4,11701163404,11703164307,0.000000,0.000000


In [9]:
predicted_flows_sydney=predicted_df

In [10]:
predicted_flows_sydney.to_csv('Sydney_OD_hybrid_predicted_Transfered_Sydney_NegBin_Seattle.csv', index=False)